<a href="https://colab.research.google.com/github/vEEr6057/image_bot/blob/main/colab_setup_web.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Enhancer Web Backend Setup\nRun this notebook to start the backend server for the Web UI.\n\n## Steps:\n1. Runtime > Change runtime type > T4 GPU\n2. Run all cells\n3. Copy the ngrok URL\n4. Add it to your Vercel environment variables

In [23]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ No GPU! Go to Runtime > Change runtime type > T4 GPU")

PyTorch: 2.9.0+cu126
CUDA: True
GPU: Tesla T4
Memory: 14.74 GB


In [2]:
!git clone https://github.com/vEEr6057/image_bot.git
%cd image_bot
!pwd
!ls -la
print("\n✅ Repository cloned!")

fatal: destination path 'image_bot' already exists and is not an empty directory.
/content/image_bot
/content/image_bot
total 160
drwxr-xr-x 8 root root  4096 Nov 25 19:36 .
drwxr-xr-x 1 root root  4096 Nov 25 19:30 ..
-rw-r--r-- 1 root root  6701 Nov 25 19:30 api_server_colab.py
-rw-r--r-- 1 root root  3677 Nov 25 19:30 api_server.py
-rw-r--r-- 1 root root  3124 Nov 25 19:30 api_server_web.py
-rw-r--r-- 1 root root 27547 Nov 25 19:30 colab_api_setup.ipynb
-rw-r--r-- 1 root root  3189 Nov 25 19:30 COLAB_GUIDE.md
-rw-r--r-- 1 root root  6833 Nov 25 19:30 colab_setup.ipynb
-rw-r--r-- 1 root root  2167 Nov 25 19:30 colab_setup_web.ipynb
-rw-r--r-- 1 root root  5561 Nov 25 19:30 CONNECT_BACKEND.md
-rw-r--r-- 1 root root   306 Nov 25 19:30 .env.example
drwxr-xr-x 8 root root  4096 Nov 25 19:30 .git
-rw-r--r-- 1 root root   173 Nov 25 19:30 .gitignore
drwxr-xr-x 8 root root  4096 Nov 25 19:39 image_bot
-rw-r--r-- 1 root root   575 Nov 25 19:30 main.py
-rw-r--r-- 1 root root  1526 Nov 25 19:3

In [6]:
# Install packages with compatible versions
!pip install --upgrade opencv-python-headless>=4.9.0.80
!pip install 'Pillow<12.0,>=8.0'
!pip install requests==2.32.4
!pip install flask flask-cors
!pip install pyngrok
print("\n✅ Basic packages installed!")

# Install Real-ESRGAN
print("\n📦 Installing Real-ESRGAN...")
!pip install basicsr facexlib gfpgan realesrgan
print("\n✅ Real-ESRGAN installed!")

# Apply compatibility patch
print("\n🔧 Applying compatibility patch...")
import sys
import torchvision.transforms.functional as F

class FunctionalTensorModule:
    @staticmethod
    def rgb_to_grayscale(img, num_output_channels=1):
        return F.rgb_to_grayscale(img, num_output_channels)

sys.modules['torchvision.transforms.functional_tensor'] = FunctionalTensorModule()
print("✅ Patch applied!")

# Verify imports
print("\n🔍 Verifying imports...")
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
print("✅ All imports successful!")


✅ Basic packages installed!

📦 Installing Real-ESRGAN...

✅ Real-ESRGAN installed!

🔧 Applying compatibility patch...
✅ Patch applied!

🔍 Verifying imports...
✅ All imports successful!


In [5]:
# Create .env file for API server
with open('.env', 'w') as f:
    f.write("USE_GPU=true\n")
    f.write("TILE_SIZE=1024\n")
    f.write("TILE_PAD=64\n")
    f.write("PRE_PAD=10\n")
    f.write("USE_FP16=true\n")

print("✅ Environment configured!")
print("GPU optimizations: tile=1024, tile_pad=64, FP16=true")

✅ Environment configured!
GPU optimizations: tile=1024, tile_pad=64, FP16=true


In [24]:
from pyngrok import ngrok

# Set ngrok authtoken
NGROK_AUTH_TOKEN = "35gxFmvmgMgzqe5Sqij0FNMI6hP_j6BMUX2ioR3wNLj5DW7"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✅ ngrok configured!")

✅ ngrok configured!


In [3]:
import threading
import time
from pyngrok import ngrok
import subprocess
import os
import signal

# --- Start: Fix for Address already in use error ---
def kill_process_on_port(port):
    try:
        # Find PIDs using the port (works on Linux/Colab)
        command = f"lsof -t -i:{port}"
        pids_str = subprocess.check_output(command, shell=True, text=True).strip()
        pids = [int(p) for p in pids_str.split()] if pids_str else []

        if pids:
            print(f"Found processes using port {port}: {pids}. Attempting to terminate them...")
            for pid in pids:
                try:
                    os.kill(pid, signal.SIGKILL) # Forcefully kill the process
                    print(f"Killed process {pid}")
                except ProcessLookupError:
                    print(f"Process {pid} already terminated.")
            time.sleep(2) # Give system time to release the port
            return True
        else:
            print(f"No process found using port {port}.")
            return False
    except Exception as e:
        print(f"Error checking/killing process on port {port}: {e}")
        print("Please consider restarting the Colab runtime if the issue persists.")
        return False

SERVER_PORT = 5000
kill_process_on_port(SERVER_PORT)
# --- End: Fix for Address already in use error ---

# Import Flask app
print("📦 Loading Flask app...")
from api_server_web import app
print("✅ Flask app loaded!")

# Start Flask in background thread
def run_flask():
    app.run(host='0.0.0.0', port=SERVER_PORT, debug=False, use_reloader=False)

print("\n🚀 Starting Flask server...")
flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

# Wait for Flask to start
time.sleep(5)
print("✅ Flask server started!")

# Create ngrok tunnel
print("\n🌐 Creating ngrok tunnel...")
public_url = ngrok.connect(SERVER_PORT)

print("\n" + "="*60)
print("🎉 API SERVER IS RUNNING!")
print("="*60)
print(f"\n📡 Public URL: {public_url}")
print("\n📋 Add this to Vercel Environment Variables:")
print(f"   NEXT_PUBLIC_BACKEND_BASE_URL={public_url}")
print("\n" + "="*60)
print("\n⚠️  IMPORTANT: Keep this notebook running!")
print("   The API will stop if you close this tab.")
print("\n🧪 Test your API:")
print(f"   {public_url}/health")
print("="*60 + "\n")

# Keep running
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Server stopped")
    ngrok.disconnect(public_url)

Error checking/killing process on port 5000: Command 'lsof -t -i:5000' returned non-zero exit status 1.
Please consider restarting the Colab runtime if the issue persists.
📦 Loading Flask app...
Model not found at /content/image_bot/weights/RealESRGAN_x4plus.pth
This may take several minutes (~65 MB)...
Downloading: 100.0% (63.9 MB / 63.9 MB)
Model downloaded successfully to /content/image_bot/weights/RealESRGAN_x4plus.pth
Model loaded: RealESRGAN_x4plus on GPU
Settings: tile=1024, tile_pad=64, pre_pad=10, half=True
✅ Flask app loaded!

🚀 Starting Flask server...
 * Serving Flask app 'api_server_web'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


✅ Flask server started!

🌐 Creating ngrok tunnel...

🎉 API SERVER IS RUNNING!

📡 Public URL: NgrokTunnel: "https://impleadable-mouthily-julien.ngrok-free.dev" -> "http://localhost:5000"

📋 Add this to Vercel Environment Variables:
   NEXT_PUBLIC_BACKEND_BASE_URL=NgrokTunnel: "https://impleadable-mouthily-julien.ngrok-free.dev" -> "http://localhost:5000"


⚠️  IMPORTANT: Keep this notebook running!
   The API will stop if you close this tab.

🧪 Test your API:
   NgrokTunnel: "https://impleadable-mouthily-julien.ngrok-free.dev" -> "http://localhost:5000"/health


🛑 Server stopped


PyngrokNgrokURLError: ngrok client exception, URLError: [Errno 111] Connection refused